# specs_novel — Phase 0, G0.1 Kaggle leg: offline DreaMS + selfies

**Generated by `build/mk_novel_p0.py` — do not hand-edit.**

Settings: **GPU T4 ×2**, **Internet OFF**. Input: the dataset **casmi26-novel-offline**
(uploaded from `_novel/p0/kaggle_upload/` by `casmi_novel_p0_colab.ipynb`).

It proves what P4 depends on: the selfies wheel installs with no network (`T_novel_install`), the
vendored DreaMS loads `dreams_embed.pt` with its SHA1 asserted, and on `cuda:1` it reproduces
Colab's vendored vectors on the 200 parity spectra (cosine ≥ 0.999 on every row, `T_embed`). The
tracker walk is repeated on 200 structures as a smoke check (recorded, not gating). ~5 min.

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np

def find_file(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True), key=len)
    assert hits, f"{name} not found under /kaggle/input -- is casmi26-novel-offline attached?"
    return hits[0]

UP = Path(find_file("manifest.json")).parent
MAN = json.loads((UP / "manifest.json").read_text())
print(f"dataset {UP}\nmanifest {json.dumps(MAN, indent=1)}")
print(subprocess.run("nvidia-smi --query-gpu=index,name,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
RESULT = {"python": sys.version.split()[0], "started": time.strftime("%Y-%m-%d %H:%M")}

t0 = time.time()
wheels = sorted(glob.glob(str(UP / "wheels" / "*.whl")))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", *wheels],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
RESULT["T_novel_install_s"] = round(time.time() - t0, 1)
RESULT["install_ok"] = r.returncode == 0
assert RESULT["install_ok"], "offline selfies install failed"
import importlib.metadata
import selfies as sf
import torch
RESULT["selfies"] = importlib.metadata.version("selfies")    # selfies 2.1.2's __version__ still says 2.1.1
RESULT["torch"] = torch.__version__
assert RESULT["selfies"] == MAN["selfies"], (RESULT["selfies"], MAN["selfies"])
print(RESULT)

## Project sources (embedded verbatim from `build/`, checked against the Colab manifest)

In [ ]:
DREAMS_INFER_SRC = '"""DreaMS embedding inference, vendored: torch + numpy only (specs_novel plan §1.3 A2, roadmap P0.1).\n\nThe upstream package (pluskal-lab/DreaMS @ dbec3a0, local `DreamS/`) cannot be imported on Kaggle:\nits pins have no Python 3.12 wheels, `dreams.api` pulls in wandb / pyopenms / umap / igraph, and its\ncheckpoints pickle `dreams.*` classes. So the network code needed for the forward pass is copied here\nwith the SAME module and attribute names, which makes the upstream `state_dict` keys load `strict`.\n\n    DreamS/dreams/models/layers/fourier_features.py   FourierFeatures            (verbatim)\n    DreamS/dreams/models/layers/feed_forward.py       FeedForward                (verbatim; interpolate_interval inlined)\n    DreamS/dreams/models/dreams/layers.py             MultiheadAttention, TFeedForward (upstream\n                                                      name FeedForward), ScaleNorm, TransformerEncoder\n    DreamS/dreams/models/dreams/dreams.py             DreaMSBackbone: DreaMS.__init__ / forward /\n                                                      __normalize_spec, inference branches only\n    DreamS/dreams/models/heads/heads.py               ContrastiveHead: head(backbone(spec)[:, 0])\n\nThe weights come from `dreams_embed.pt`, written once on Colab by the P0.1a conversion (the real\npackage loads `embedding_model.ckpt` + `ssl_model.ckpt`; `args_to_dict` + `state_dict` are saved).\nFile layout: {"args_json": str, "state_dict": {name: tensor}, "meta_json": str}; plain types only, so\n`torch.load(weights_only=True)` works.\n\nPreprocessing is split in two, exactly as the cached corpus was built:\n    dreams_peaks     casmi_dreams_colab.ipynb cell 19 `_dreams_peaks` (= cell 7 `peak_list`): +2 Da cut,\n                     0.2 % floor, top-100, m/z ascending, intensity / max   -> (2, n) float32\n    to_model_input   upstream SpectrumPreprocessor(DataFormatA, n_highest_peaks=100) as used by\n                     `dreams_embeddings`: trim, zero-pad to 100, relative intensity, prepend\n                     [precursor_mz, 1.1], float32                           -> (101, 2) float32\nm/z above 1000 is NOT clipped (upstream never sets normalize_mzs); it is only outside pretraining.\n"""\nimport hashlib\nimport json\nfrom math import ceil\nfrom types import SimpleNamespace\nfrom typing import Sequence\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torch.nn import Parameter\n\nN_PEAKS = 100             # checkpoint\'s n_highest_peaks (dreams/api.py `from_name`)\nPREC_INTENS = 1.1         # SpectrumPreprocessor default\nINT_FLOOR = 0.002         # casmi_dreams_colab.ipynb CFG\nDROP_ABOVE_PRECURSOR = 2.0\nMAX_MZ = 1000.0           # dreams/utils/dformats.py DataFormatA.max_mz\nMAX_TBXIC_STDEV = 1e-4    # DataFormatA.max_tbxic_stdev\nTRAINING_ONLY_PREFIXES = ("backbone.ff_out", "backbone.ff_out_intens", "backbone.mz_masking_loss",\n                          "backbone.ro_out")\n\n# Backbone args the inference path reads (dreams.py __init__, layers.py). Anything else is ignored.\nARG_KEYS = ("n_layers", "n_heads", "d_fourier", "d_peak", "d_mz_token", "charge_feature",\n            "fourier_strategy", "fourier_num_freqs", "fourier_min_freq", "fourier_trainable",\n            "dropout", "ff_fourier_depth", "ff_fourier_d", "no_ffs_bias", "ff_peak_depth",\n            "vanilla_transformer", "graphormer_mz_diffs", "graphormer_parametrized", "att_dropout",\n            "no_transformer_bias", "attn_mech", "ff_dropout", "residual_dropout", "pre_norm", "scnorm",\n            "dformat_max_mz", "dformat_max_tbxic_stdev")\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --------------------------------------------------------------------------------------------- #\n# Preprocessing\n# --------------------------------------------------------------------------------------------- #\n\ndef dreams_peaks(mz, it, prec, floor=INT_FLOOR, n=N_PEAKS, above=DROP_ABOVE_PRECURSOR):\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    if len(mz) == 0: return None\n    k = mz <= prec + above; mz, it = mz[k], it[k]\n    if len(mz) == 0: return None\n    mx = it.max()\n    if not np.isfinite(mx) or mx <= 0: return None\n    k = it >= floor*mx; mz, it = mz[k], it[k]\n    if len(mz) < 2: return None\n    # Deterministic where cell 7 was not: its np.argsort (unstable) let the numpy build / CPU decide which\n    # tied peaks survive the top-n cut (G0.1-R). Ties at the cut go to the higher m/z; order is by m/z,\n    # then intensity, so the output does not depend on the input order either.\n    if len(mz) > n:\n        keep = np.lexsort((mz, it))[-n:]; mz, it = mz[keep], it[keep]\n    o = np.lexsort((it, mz)); mz, it = mz[o], it[o]\n    return np.stack([mz, it/it.max()]).astype(np.float32)   # (2, n)\n\n\ndef to_model_input(peaks, prec_mz, n=N_PEAKS, prec_intens=PREC_INTENS):\n    """(2, k) peak list -> (n + 1, 2) float32: upstream SpectrumPreprocessor.__call__ with its defaults."""\n    pl = np.array(peaks, copy=True)                                  # (2, k)\n    keep = np.sort(np.lexsort((pl[0], pl[1]))[-n:])                  # su.trim_peak_list, ties -> higher m/z\n    pl = pl[:, keep]\n    if pl.shape[1] < n:                                              # su.pad_peak_list\n        pl = np.pad(pl, ((0, 0), (0, n - pl.shape[1])), mode="constant", constant_values=0)\n    pl = np.array([pl[0], pl[1] / pl[1].max()])                      # su.to_rel_intensity\n    spec = np.vstack([np.array([prec_mz, prec_intens]), pl.T])       # su.prepend_precursor_peak\n    return spec.astype(np.float32, copy=False)\n\n\n# --------------------------------------------------------------------------------------------- #\n# Network (upstream code; see the module docstring for the file of origin)\n# --------------------------------------------------------------------------------------------- #\n\ndef interpolate_interval(a, b, n, only_inter=False, rounded=False):\n    x_min, x_max = min(a, b), max(a, b)\n    res = [x_min + i * (x_max - x_min) / (n + 1) for i in range(1 if only_inter else 0, n + 1 if only_inter else n + 2)]\n    if x_max == a:\n        res.reverse()\n    if rounded:\n        res = [round(x) for x in res]\n    return res\n\n\nclass FourierFeatures(nn.Module):\n    def __init__(self, strategy, x_min, x_max, trainable=True, funcs=\'both\', sigma=10, num_freqs=512):\n\n        assert strategy in {\'random\', \'voronov_et_al\', \'lin_float_int\'}\n        assert funcs in {\'both\', \'sin\', \'cos\'}\n        assert x_min < 1\n\n        super().__init__()\n        self.funcs = funcs\n        self.strategy = strategy\n        self.trainable = trainable\n        self.num_freqs = num_freqs\n\n        if strategy == \'random\':\n            self.b = torch.randn(num_freqs) * sigma\n        if self.strategy == \'voronov_et_al\':\n            self.b = torch.tensor(\n                [1 / (x_min * (x_max / x_min) ** (2 * i / (num_freqs - 2))) for i in range(1, num_freqs)],\n            )\n        elif self.strategy == \'lin_float_int\':\n            self.b = torch.tensor(\n                [1 / (x_min * i) for i in range(2, ceil(1 / x_min), 2)] +\n                [1 / (1 * i) for i in range(2, ceil(x_max), 1)],\n            )\n        self.b = self.b.unsqueeze(0)\n\n        self.b = nn.Parameter(self.b, requires_grad=self.trainable)\n        self.register_parameter(\'Fourier frequencies\', self.b)\n\n    def forward(self, x):\n        x = 2 * torch.pi * x @ self.b\n        if self.funcs == \'both\':\n            x = torch.cat((torch.cos(x), torch.sin(x)), dim=-1)\n        elif self.funcs == \'cos\':\n            x = torch.cos(x)\n        elif self.funcs == \'sin\':\n            x = torch.sin(x)\n        return x\n\n    def num_features(self):\n        return self.b.shape[1] if self.funcs != \'both\' else 2 * self.b.shape[1]\n\n\nclass FeedForward(nn.Module):\n\n    def __init__(self, in_dim, out_dim, hidden_dim, depth=None, act_last=True, act=nn.ReLU, bias=True, dropout=0):\n        super().__init__()\n\n        if isinstance(hidden_dim, int):\n            assert depth is not None\n            hidden_dim = [hidden_dim] * depth\n        elif hidden_dim == \'interpolated\':\n            assert depth is not None\n            hidden_dim = interpolate_interval(a=in_dim, b=out_dim, n=depth - 1, only_inter=True, rounded=True)\n        elif isinstance(hidden_dim, Sequence):  # e.g. is List or Tuple\n            depth = len(hidden_dim)\n        else:\n            raise ValueError\n\n        self.ff = nn.ModuleList([])\n        for l in range(depth):\n            d1 = hidden_dim[l - 1] if l != 0 else in_dim\n            d2 = hidden_dim[l] if l != depth - 1 else out_dim\n            self.ff.append(nn.Linear(d1, d2, bias=bias))\n            if l != depth - 1:\n                self.ff.append(nn.Dropout(p=dropout))\n            if l != depth - 1 or act_last:\n                self.ff.append(act())\n        self.ff = nn.Sequential(*self.ff)\n\n    def forward(self, x):\n        return self.ff(x)\n\n\nclass MultiheadAttention(nn.Module):\n    def __init__(self, args):\n        super(MultiheadAttention, self).__init__()\n        self.d_model = args.d_model\n        self.n_heads = args.n_heads\n        self.dropout = args.att_dropout\n        self.use_transformer_bias = not args.no_transformer_bias\n        self.attn_mech = args.attn_mech\n        self.d_graphormer_params = args.d_graphormer_params\n\n        if self.d_model % self.n_heads != 0:\n            raise ValueError(\'Required: d_model % n_heads == 0.\')\n\n        self.head_dim = self.d_model // self.n_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.weights = Parameter(torch.Tensor(4 * self.d_model, self.d_model))\n        if self.use_transformer_bias:\n            self.biases = Parameter(torch.Tensor(4 * self.d_model))\n\n        if self.d_graphormer_params:\n            self.lin_graphormer = nn.Linear(self.d_graphormer_params, self.n_heads, bias=False)\n\n        mean = 0\n        std = (2 / (5 * self.d_model)) ** 0.5\n        nn.init.normal_(self.weights, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.biases, 0.)\n\n        if self.attn_mech == \'additive_v\':\n            self.additive_v = Parameter(torch.Tensor(self.n_heads, self.head_dim))\n            nn.init.normal_(self.additive_v, mean=mean, std=std)\n\n    def forward(self, q, k, v, mask, graphormer_dists=None, do_proj_qkv=True):\n        bs, n, d = q.size()\n\n        def _split_heads(tensor):\n            bsz, length, d_model = tensor.size()\n            return tensor.reshape(bsz, length, self.n_heads, self.head_dim).transpose(1, 2)\n\n        if do_proj_qkv:\n            q, k, v = self.proj_qkv(q, k, v)\n\n        q = _split_heads(q)\n        k = _split_heads(k)\n        v = _split_heads(v)\n\n        if self.attn_mech == \'dot-product\':\n            att_weights = torch.einsum(\'bhnd,bhdm->bhnm\', q, k.transpose(-2, -1))\n        elif self.attn_mech == \'additive_v\' or self.attn_mech == \'additive_fixed\':\n            att_weights = (q.unsqueeze(-2) - k.unsqueeze(-3))\n            if self.attn_mech == \'additive_v\':\n                att_weights = (att_weights * self.additive_v.unsqueeze(0).unsqueeze(2).unsqueeze(3))\n            att_weights = att_weights.sum(dim=-1)\n        else:\n            raise NotImplementedError(f\'"{self.attn_mech}" attention mechanism is not implemented.\')\n        att_weights = att_weights * self.scale\n\n        if graphormer_dists is not None:\n            if self.d_graphormer_params:\n                att_bias = self.lin_graphormer(graphormer_dists).permute(0, 3, 1, 2)\n            else:\n                att_bias = graphormer_dists.sum(dim=-1).unsqueeze(1)\n            att_weights = att_weights + att_bias\n\n        if mask is not None:\n            att_weights.masked_fill_(mask.unsqueeze(1).unsqueeze(-1), -1e9)\n\n        att_weights = F.softmax(att_weights, dim=-1)\n        att_weights = F.dropout(att_weights, p=self.dropout, training=self.training)\n        _att_weights = att_weights.reshape(-1, n, n)\n        output = torch.bmm(_att_weights, v.reshape(bs * self.n_heads, -1, self.head_dim))\n        output = output.reshape(bs, self.n_heads, n, self.head_dim).transpose(1, 2).reshape(bs, n, -1)\n        output = self.proj_o(output)\n\n        return output, att_weights\n\n    def proj_qkv(self, q, k, v):\n        qkv_same = q.data_ptr() == k.data_ptr() == v.data_ptr()\n        kv_same = k.data_ptr() == v.data_ptr()\n\n        if qkv_same:\n            q, k, v = self._proj(q, end=3 * self.d_model).chunk(3, dim=-1)\n        elif kv_same:\n            q = self._proj(q, end=self.d_model)\n            k, v = self._proj(k, start=self.d_model, end=3 * self.d_model).chunk(2, dim=-1)\n        else:\n            q = self.proj_q(q)\n            k = self.proj_k(k)\n            v = self.proj_v(v)\n\n        return q, k, v\n\n    def _proj(self, x, start=0, end=None):\n        weight = self.weights[start:end, :]\n        bias = None if not self.use_transformer_bias else self.biases[start:end]\n        return F.linear(x, weight=weight, bias=bias)\n\n    def proj_q(self, q):\n        return self._proj(q, end=self.d_model)\n\n    def proj_k(self, k):\n        return self._proj(k, start=self.d_model, end=2 * self.d_model)\n\n    def proj_v(self, v):\n        return self._proj(v, start=2 * self.d_model, end=3 * self.d_model)\n\n    def proj_o(self, x):\n        return self._proj(x, start=3 * self.d_model)\n\n\nclass TFeedForward(nn.Module):\n    """Upstream `dreams.models.dreams.layers.FeedForward` (renamed: clashes with the peak FeedForward)."""\n    def __init__(self, args):\n        super(TFeedForward, self).__init__()\n        self.dropout = args.ff_dropout\n        self.d_model = args.d_model\n        self.ff_dim = 4 * args.d_model\n        self.use_transformer_bias = not args.no_transformer_bias\n\n        self.in_proj = nn.Linear(self.d_model, self.ff_dim, bias=self.use_transformer_bias)\n        self.out_proj = nn.Linear(self.ff_dim, self.d_model, bias=self.use_transformer_bias)\n\n        mean = 0\n        std = (2 / (self.ff_dim + self.d_model)) ** 0.5\n        nn.init.normal_(self.in_proj.weight, mean=mean, std=std)\n        nn.init.normal_(self.out_proj.weight, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.in_proj.bias, 0.)\n            nn.init.constant_(self.out_proj.bias, 0.)\n\n    def forward(self, x):\n        y = F.relu(self.in_proj(x))\n        y = F.dropout(y, p=self.dropout, training=self.training)\n        return self.out_proj(y)\n\n\nclass ScaleNorm(nn.Module):\n    def __init__(self, scale, eps=1e-5):\n        super(ScaleNorm, self).__init__()\n        self.scale = Parameter(torch.tensor(scale))\n        self.eps = eps\n\n    def forward(self, x):\n        norm = self.scale / torch.norm(x, dim=-1, keepdim=True).clamp(min=self.eps)\n        return x * norm\n\n\nclass TransformerEncoder(nn.Module):\n    def __init__(self, args):\n        super(TransformerEncoder, self).__init__()\n        self.residual_dropout = args.residual_dropout\n        self.n_layers = args.n_layers\n        self.pre_norm = args.pre_norm\n\n        self.atts = nn.ModuleList([MultiheadAttention(args) for _ in range(self.n_layers)])\n        self.ffs = nn.ModuleList([TFeedForward(args) for _ in range(self.n_layers)])\n\n        num_scales = self.n_layers * 2 + 1 if self.pre_norm else self.n_layers * 2\n        if args.scnorm:\n            self.scales = nn.ModuleList([ScaleNorm(args.d_model ** 0.5) for _ in range(num_scales)])\n        else:\n            self.scales = nn.ModuleList([nn.LayerNorm(args.d_model) for _ in range(num_scales)])\n\n    def _layer_forward(self, i, x, src_mask, graphormer_dists):\n        pre_norm = self.pre_norm\n        post_norm = not pre_norm\n        att = self.atts[i]\n        ff = self.ffs[i]\n        att_scale = self.scales[2 * i]\n        ff_scale = self.scales[2 * i + 1]\n\n        residual = x\n        x = att_scale(x) if pre_norm else x\n        x, _ = att(q=x, k=x, v=x, mask=src_mask, graphormer_dists=graphormer_dists)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = att_scale(x) if post_norm else x\n\n        residual = x\n        x = ff_scale(x) if pre_norm else x\n        x = ff(x)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = ff_scale(x) if post_norm else x\n        return x\n\n    def forward(self, src_inputs, src_mask, graphormer_dists=None):\n        x = F.dropout(src_inputs, p=self.residual_dropout, training=self.training)\n        for i in range(self.n_layers):\n            x = self._layer_forward(i, x, src_mask, graphormer_dists)\n        x = self.scales[-1](x) if self.pre_norm else x\n        return x\n\n\nclass DreaMSBackbone(nn.Module):\n    """Upstream `DreaMS` (LightningModule) reduced to the modules the embedding forward pass uses."""\n\n    def __init__(self, args):\n        super().__init__()\n        if args.charge_feature:\n            raise NotImplementedError("charge_feature=True: the embedding checkpoint is used without charge")\n        if args.d_mz_token or not args.d_fourier:\n            raise NotImplementedError("only the Fourier-feature m/z encoding is vendored")\n        if args.vanilla_transformer:\n            raise NotImplementedError("only the DreaMS TransformerEncoder is vendored")\n        self.n_layers = args.n_layers\n        self.d_fourier = args.d_fourier\n        self.d_peak = args.d_peak\n        self.d_model = sum(d for d in [args.d_fourier, args.d_peak, args.d_mz_token] if d)\n        args.d_model = self.d_model\n        self.max_mz = args.dformat_max_mz\n        self.graphormer_mz_diffs = args.graphormer_mz_diffs\n        if args.graphormer_mz_diffs and args.graphormer_parametrized:\n            args.d_graphormer_params = args.d_fourier if args.d_fourier else 1\n        else:\n            args.d_graphormer_params = 0\n\n        token_dim = 2\n        self.fourier_enc = FourierFeatures(strategy=args.fourier_strategy, num_freqs=args.fourier_num_freqs,\n                                           x_min=args.dformat_max_tbxic_stdev if not args.fourier_min_freq else args.fourier_min_freq,\n                                           x_max=args.dformat_max_mz,\n                                           trainable=args.fourier_trainable)\n        self.ff_fourier = FeedForward(in_dim=self.fourier_enc.num_features(), out_dim=args.d_fourier, dropout=args.dropout,\n                                      depth=args.ff_fourier_depth, hidden_dim=args.ff_fourier_d, bias=not args.no_ffs_bias)\n        self.ff_peak = FeedForward(in_dim=token_dim, hidden_dim=args.d_peak, out_dim=args.d_peak, depth=args.ff_peak_depth,\n                                   dropout=args.dropout, bias=not args.no_ffs_bias)\n        self.transformer_encoder = TransformerEncoder(args)\n\n    def forward(self, spec):\n        padding_mask = spec[:, :, 0] == 0\n        peak_embs = self.ff_peak(spec / spec.new_tensor([self.max_mz, 1.]))\n        fourier_features = self.ff_fourier(self.fourier_enc(spec[..., [0]]))\n        spec = torch.cat([peak_embs, fourier_features], dim=-1)\n        graphormer_dists = None\n        if self.graphormer_mz_diffs:\n            graphormer_dists = fourier_features.unsqueeze(2) - fourier_features.unsqueeze(1)\n        return self.transformer_encoder(spec, padding_mask, graphormer_dists)\n\n\nclass DreamsEmbedder(nn.Module):\n    """Upstream ContrastiveHead at inference: Linear(d_model, d_model) on the precursor token."""\n\n    def __init__(self, args):\n        super().__init__()\n        self.backbone = DreaMSBackbone(args)\n        self.head = nn.Linear(self.backbone.d_model, self.backbone.d_model, bias=True)\n\n    def forward(self, spec):\n        return self.head(self.backbone(spec)[:, 0, ...])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Conversion (Colab, with the real package loaded) and loading (anywhere)\n# --------------------------------------------------------------------------------------------- #\n\ndef args_to_dict(args):\n    """Upstream backbone `args` (argparse Namespace; `dformat` is a DataFormatA object) -> plain dict."""\n    src = vars(args) if not isinstance(args, dict) else args\n    out = {k: src[k] for k in ARG_KEYS if k in src}\n    dformat = src.get("dformat")\n    out["dformat_max_mz"] = float(getattr(dformat, "max_mz", MAX_MZ))\n    out["dformat_max_tbxic_stdev"] = float(getattr(dformat, "max_tbxic_stdev", MAX_TBXIC_STDEV))\n    missing = [k for k in ARG_KEYS if k not in out]\n    assert not missing, f"backbone args lack {missing}"\n    return out\n\n\ndef inference_state_dict(state_dict):\n    """Drop the training-only backbone heads that PreTrainedModel deletes after loading (api.py)."""\n    return {k: v for k, v in state_dict.items() if not k.startswith(TRAINING_ONLY_PREFIXES)}\n\n\ndef save_dreams(path, args, state_dict, meta):\n    torch.save({"args_json": json.dumps(args_to_dict(args), sort_keys=True),\n                "state_dict": {k: v.detach().cpu().float().contiguous() for k, v in\n                               inference_state_dict(state_dict).items()},\n                "meta_json": json.dumps(meta, sort_keys=True)}, path)\n\n\ndef build_embedder(args_dict):\n    return DreamsEmbedder(SimpleNamespace(**args_dict))\n\n\ndef load_dreams(path, device="cpu", expect_sha1=None):\n    """-> (DreamsEmbedder in eval / float32 on device, meta dict). strict load: every key must match."""\n    if expect_sha1 is not None:\n        got = sha1(path)\n        assert got == expect_sha1, f"{path}: sha1 {got} != expected {expect_sha1}"\n    blob = torch.load(path, map_location="cpu", weights_only=True)\n    model = build_embedder(json.loads(blob["args_json"]))\n    model.load_state_dict(blob["state_dict"], strict=True)\n    return model.to(device=device, dtype=torch.float32).eval(), json.loads(blob["meta_json"])\n\n\n@torch.inference_mode()\ndef dreams_embed(model, peak_lists, precursors, batch_size=64):\n    """peak_lists: (2, k) arrays from `dreams_peaks`; precursors: m/z. -> (n, d_model) float32 numpy."""\n    dev = next(model.parameters()).device\n    out = []\n    for lo in range(0, len(peak_lists), batch_size):\n        x = np.stack([to_model_input(p, m) for p, m in\n                      zip(peak_lists[lo:lo + batch_size], precursors[lo:lo + batch_size])])\n        out.append(model(torch.from_numpy(x).to(dev)).float().cpu().numpy())\n    d = model.head.out_features\n    return np.concatenate(out) if out else np.zeros((0, d), np.float32)\n'
NOVEL_DECODER_SRC = '"""specs_novel single source of truth for the SELFIES side (plan §4, §5.5). P0 part: no model yet.\n\n    SelfiesVocab        token <-> id, the same ids as MassSpecGym\'s SelfiesTokenizer (JSON, no `tokenizers`)\n    TokenTable          per-token grammar facts, precomputed with selfies\' own grammar_rules\n    Tracker             incremental SELFIES derivation state -> legal-token mask (plan §4.1-4.2)\n    reference_state     the slow truth: selfies.decoder(prefix) + RDKit counts (P0.4 parity)\n    walk_truth          mask soundness: every true next token legal, </s> legal at the end (P0.4)\n    prep_structure      stereo/isotope strip -> canonical SMILES -> SELFIES (A9, P0.3)\n\nThe Tracker replays `selfies.decoder._derive_mol_from_symbols` token by token. That function is\nrecursive (a branch derives a sub-chain of Q+1 symbols); here its call stack is an explicit list of\nframes [state, prev_atom, budget]. Every consumed symbol lowers the budget of every open frame, the\nway each caller adds its child\'s n_derived; a frame closes when its budget reaches 0. A frame whose\nstate becomes None (no free valence left) swallows the rest of its budget, and in the main chain that\nmeans every later symbol -- SELFIES silently skips those atoms (A7), so the mask never offers them.\n\nMasks (plan §4.2):\n    pending index symbols      only the 16 INDEX_ALPHABET symbols (the encoder emits nothing else there)\n    atom token, element e      frame active, the atom would be placed, and remaining[e] > 0\n    [Branch*]                  frame state > 1 (otherwise selfies ignores it)\n    [Ring*]                    frame state > 0\n    </s>                       no pending index and every heavy-atom count exactly used\n`unsat()` is rings + pi bonds of the molecule the prefix decodes to, with the ring bonds formed the way\n`_form_rings_bilocally` forms them at the end (orders capped by the free valence left). It feeds the\noptional DBE prune, which is only applied where P0.4 shows it sound (A8).\n\nNeeds numpy + selfies (pinned); RDKit only for reference_state / prep_structure / ik14.\n"""\nimport json\nimport re\n\nimport numpy as np\nimport selfies as sf\nfrom selfies.constants import INDEX_CODE\nfrom selfies.grammar_rules import (get_index_from_selfies, next_atom_state, next_branch_state,\n                                   next_ring_state, process_atom_symbol, process_branch_symbol,\n                                   process_ring_symbol)\n\nPAD, SOS, EOS, UNK = "<pad>", "<s>", "</s>", "<unk>"\nALLOWED_ELEMENTS = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")\nDBE_SAFE_ELEMENTS = frozenset(("C", "H", "N", "O", "F", "Cl", "Br", "I"))   # no S / P (plan §4.2)\nK_SPECIAL, K_ATOM, K_BRANCH, K_RING, K_OTHER = 0, 1, 2, 3, 4\n_FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef check_constraints():\n    """The token table and the tracker assume selfies\' default semantic constraints."""\n    assert sf.get_semantic_constraints() == sf.get_preset_constraints("default"), \\\n        "selfies semantic constraints are not the default preset"\n\n\n# --------------------------------------------------------------------------------------------- #\n# Vocabulary\n# --------------------------------------------------------------------------------------------- #\n\ndef vocab_from_alphabet(alphabet):\n    """SelfiesTokenizer\'s id layout (massspecgym/models/tokenizers.py:79-85 + add_special_tokens):\n    sorted alphabet 0..n-1, <unk> n, then <pad> <s> </s> as the next added tokens."""\n    alphabet = sorted(set(alphabet) - {PAD, SOS, EOS, UNK})\n    tok2id = {s: i for i, s in enumerate(alphabet)}\n    tok2id[UNK] = len(tok2id)\n    for s in (PAD, SOS, EOS):\n        tok2id[s] = len(tok2id)\n    return tok2id\n\n\nclass SelfiesVocab:\n    def __init__(self, tok2id):\n        ids = sorted(tok2id.values())\n        assert ids == list(range(len(ids))), "token ids must be 0..n-1"\n        for s in (PAD, SOS, EOS, UNK):\n            assert s in tok2id, f"special token {s} missing"\n        self.tok2id = dict(tok2id)\n        self.id2tok = [None] * len(ids)\n        for s, i in self.tok2id.items():\n            self.id2tok[i] = s\n        self.pad_id, self.sos_id, self.eos_id, self.unk_id = (self.tok2id[s] for s in (PAD, SOS, EOS, UNK))\n\n    def __len__(self):\n        return len(self.id2tok)\n\n    @classmethod\n    def from_alphabet(cls, alphabet):\n        return cls(vocab_from_alphabet(alphabet))\n\n    @classmethod\n    def from_json(cls, path):\n        with open(path, encoding="utf-8") as f:\n            return cls(json.load(f))\n\n    def to_json(self, path):\n        with open(path, "w", encoding="utf-8") as f:\n            json.dump(self.tok2id, f, sort_keys=True, indent=0)\n\n    def encode(self, selfies_str):\n        """SELFIES string -> ids, no <s>/</s>."""\n        return [self.tok2id.get(t, self.unk_id) for t in sf.split_selfies(selfies_str)]\n\n    def decode(self, ids):\n        """ids -> SELFIES string; special tokens dropped."""\n        return "".join(self.id2tok[i] for i in ids if i not in (self.pad_id, self.sos_id, self.eos_id,\n                                                                self.unk_id))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Token table\n# --------------------------------------------------------------------------------------------- #\n\nclass TokenTable:\n    """Per-token facts over a vocabulary, from selfies\' own grammar_rules (so they follow the pin)."""\n\n    def __init__(self, vocab):\n        check_constraints()\n        n = len(vocab)\n        self.vocab = vocab\n        self.kind = np.full(n, K_OTHER, np.int8)\n        self.elem = np.full(n, -1, np.int16)           # index into self.elements\n        self.bond = np.zeros(n, np.int8)               # requested bond order (atom / branch / ring type)\n        self.cap = np.zeros(n, np.int8)                # atom bonding capacity (after explicit H, charge)\n        self.nidx = np.zeros(n, np.int8)               # index symbols read after a branch / ring\n        self.is_index = np.zeros(n, bool)              # one of the 16 INDEX_ALPHABET symbols\n        self.index_code = np.zeros(n, np.int16)\n        elements = []\n        for i, t in enumerate(vocab.id2tok):\n            if t in (PAD, SOS, EOS, UNK):\n                self.kind[i] = K_SPECIAL\n                continue\n            if t in INDEX_CODE:\n                self.is_index[i] = True\n                self.index_code[i] = INDEX_CODE[t]\n            if "ch" == t[-4:-2]:                       # the same dispatch order as the decoder\n                out = process_branch_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_BRANCH, out[0], out[1]\n            elif "ng" == t[-4:-2]:\n                out = process_ring_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_RING, out[0], out[1]\n            elif "eps" in t:\n                pass\n            else:\n                out = process_atom_symbol(t)\n                if out is not None:\n                    (bond, _), atom = out\n                    if atom.element not in elements:\n                        elements.append(atom.element)\n                    self.kind[i] = K_ATOM\n                    self.elem[i] = elements.index(atom.element)\n                    self.bond[i] = bond\n                    self.cap[i] = atom.bonding_capacity\n        self.elements = tuple(elements)\n        self.elem_ix = {e: j for j, e in enumerate(self.elements)}\n        self.h_ix = self.elem_ix.get("H", -1)\n        self.is_atom = self.kind == K_ATOM\n        self.is_branch = self.kind == K_BRANCH\n        self.is_ring = self.kind == K_RING\n        self.atom_cap_pos = self.is_atom & (self.cap > 0)\n        self.elem_safe = np.maximum(self.elem, 0)\n\n    def counts_vector(self, counts):\n        """{element: count} -> vector over self.elements, plus the count of elements the vocab lacks."""\n        v = np.zeros(len(self.elements), np.int32)\n        missing = 0\n        for e, c in counts.items():\n            if e in self.elem_ix:\n                v[self.elem_ix[e]] = c\n            elif e != "H":\n                missing += c\n        return v, missing\n\n\n# --------------------------------------------------------------------------------------------- #\n# Tracker\n# --------------------------------------------------------------------------------------------- #\n\nINF = float("inf")\n\n\nclass Tracker:\n    """Persistent SELFIES derivation state. `push` returns a new Tracker; the old one is unchanged.\n\n    formula: {element: count} incl. H, or None (then only counting, no count masks)."""\n\n    __slots__ = ("tt", "frames", "pending", "caps", "bc", "bonds", "rings", "counts", "target",\n                 "missing", "n_tokens", "broken")\n\n    def __init__(self, table, formula=None):\n        self.tt = table\n        self.frames = [[0, None, INF]]          # [state (int | None), prev atom index, budget]\n        self.pending = None                     # [kind, n_left, symbols, payload]\n        self.caps = []                          # bonding capacity per placed atom\n        self.bc = []                            # bond order sum per atom (chain bonds only)\n        self.bonds = {}                         # (a, b) a < b -> order (chain bonds only)\n        self.rings = []                         # (left atom, right atom, order), formed at the end\n        self.counts = np.zeros(len(table.elements), np.int32)\n        if formula is None:\n            self.target, self.missing = None, 0\n        else:\n            self.target, self.missing = table.counts_vector(formula)\n        self.n_tokens = 0\n        self.broken = False\n\n    def _copy(self):\n        t = Tracker.__new__(Tracker)\n        t.tt = self.tt\n        t.frames = [f[:] for f in self.frames]\n        t.pending = None if self.pending is None else [self.pending[0], self.pending[1],\n                                                       self.pending[2][:], self.pending[3]]\n        t.caps, t.bc, t.bonds, t.rings = self.caps[:], self.bc[:], dict(self.bonds), self.rings[:]\n        t.counts = self.counts.copy()\n        t.target, t.missing, t.n_tokens, t.broken = self.target, self.missing, self.n_tokens, self.broken\n        return t\n\n    # -- derivation -------------------------------------------------------------------------- #\n\n    def _consume(self):\n        for f in self.frames:\n            f[2] -= 1\n\n    def _close_exhausted(self):\n        while len(self.frames) > 1 and self.frames[-1][2] <= 0:\n            self.frames.pop()\n\n    def _add_atom(self, cap):\n        self.caps.append(int(cap))\n        self.bc.append(0)\n        return len(self.caps) - 1\n\n    def push(self, tok_id):\n        t = self._copy()\n        t._apply(int(tok_id))\n        return t\n\n    def _apply(self, i):\n        tt = self.tt\n        self.n_tokens += 1\n        self._consume()\n        if self.pending is not None:                      # an index symbol of a branch / ring\n            p = self.pending\n            p[2].append(tt.vocab.id2tok[i])\n            p[1] -= 1\n            if p[1] == 0:\n                self.pending = None\n                q = get_index_from_selfies(*p[2])\n                f = self.frames[-1]\n                if p[0] == K_BRANCH:\n                    self.frames.append([p[3], f[1], q + 1])\n                else:\n                    lidx = max(0, f[1] - (q + 1))\n                    self.rings.append((lidx, f[1], p[3]))\n            self._close_exhausted()\n            return\n        f = self.frames[-1]\n        state = f[0]\n        kind = tt.kind[i]\n        if state is None or kind in (K_SPECIAL, K_OTHER):\n            if kind == K_OTHER and state is not None and "eps" in tt.vocab.id2tok[i]:\n                f[0] = 0 if state == 0 else None\n            elif kind == K_SPECIAL:\n                self.broken = True                        # specials never occur inside a SELFIES string\n        elif kind == K_BRANCH:\n            if state > 1:\n                binit, f[0] = next_branch_state(int(tt.bond[i]), state)\n                self.pending = [K_BRANCH, int(tt.nidx[i]), [], binit]\n        elif kind == K_RING:\n            if state != 0:\n                order, f[0] = next_ring_state(int(tt.bond[i]), state)\n                self.pending = [K_RING, int(tt.nidx[i]), [], order]\n        else:                                             # atom\n            cap = int(tt.cap[i])\n            order, nxt = next_atom_state(int(tt.bond[i]), cap, state)\n            if order == 0:\n                if state == 0:\n                    f[1] = self._add_atom(cap)\n                    self.counts[tt.elem[i]] += 1\n                else:\n                    self.broken = True                    # selfies would bond from an unplaced atom\n            else:\n                a = self._add_atom(cap)\n                self.counts[tt.elem[i]] += 1\n                src = f[1]\n                self.bonds[(src, a)] = order\n                self.bc[src] += order\n                self.bc[a] += order\n                f[1] = a\n            f[0] = nxt\n        self._close_exhausted()\n\n    # -- queries ----------------------------------------------------------------------------- #\n\n    def remaining(self):\n        return None if self.target is None else self.target - self.counts\n\n    def heavy_done(self):\n        r = self.remaining()\n        if r is None:\n            return True\n        r = r.copy()\n        if self.tt.h_ix >= 0:\n            r[self.tt.h_ix] = 0\n        return self.missing == 0 and not r.any()\n\n    def eos_legal(self):\n        return (not self.broken and self.pending is None and len(self.caps) > 0 and self.heavy_done())\n\n    def legal_mask(self):\n        """numpy bool over the vocabulary."""\n        tt = self.tt\n        n = len(tt.kind)\n        if self.broken:\n            return np.zeros(n, bool)\n        if self.pending is not None:\n            return tt.is_index.copy()\n        state = self.frames[-1][0]\n        mask = np.zeros(n, bool)\n        if state is not None:\n            placed = tt.is_atom if state == 0 else tt.atom_cap_pos\n            if self.target is not None:\n                placed = placed & (self.remaining()[tt.elem_safe] > 0)\n            mask |= placed\n            if state > 1:\n                mask |= tt.is_branch\n            if state > 0:\n                mask |= tt.is_ring\n        if self.eos_legal():\n            mask[tt.vocab.eos_id] = True\n        return mask\n\n    def unsat(self):\n        """Rings + pi bonds of the molecule this prefix decodes to (selfies\' _form_rings_bilocally)."""\n        n = len(self.caps)\n        if n == 0:\n            return 0\n        bc, bonds = self.bc[:], dict(self.bonds)\n        for l, r, order in self.rings:\n            if l == r:\n                continue\n            lfree, rfree = self.caps[l] - bc[l], self.caps[r] - bc[r]\n            if lfree <= 0 or rfree <= 0:\n                continue\n            order = min(order, lfree, rfree)\n            key = (min(l, r), max(l, r))\n            if key in bonds:\n                new = min(order + bonds[key], 3)\n                bc[l] += new - bonds[key]\n                bc[r] += new - bonds[key]\n                bonds[key] = new\n            else:\n                bonds[key] = order\n                bc[l] += order\n                bc[r] += order\n        return sum(bonds.values()) - n + 1\n\n    def element_counts(self):\n        return {e: int(c) for e, c in zip(self.tt.elements, self.counts) if c}\n\n\n# --------------------------------------------------------------------------------------------- #\n# Formulae\n# --------------------------------------------------------------------------------------------- #\n\ndef parse_formula(formula):\n    counts = {}\n    for e, c in _FORMULA_RE.findall(formula):\n        counts[e] = counts.get(e, 0) + (int(c) if c else 1)\n    return counts\n\n\ndef dbe(counts):\n    """Rings + pi bonds implied by a formula with standard valences (C4 N3 O2 X1 H1)."""\n    x = sum(counts.get(e, 0) for e in ("F", "Cl", "Br", "I"))\n    return 1 + counts.get("C", 0) - (counts.get("H", 0) + x) / 2 + counts.get("N", 0) / 2\n\n\ndef dbe_prune_applies(counts):\n    return set(e for e, c in counts.items() if c) <= DBE_SAFE_ELEMENTS\n\n\n# --------------------------------------------------------------------------------------------- #\n# RDKit side: structure prep, ik14, reference tracker\n# --------------------------------------------------------------------------------------------- #\n\ndef _chem():\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    return Chem\n\n\ndef mol_formula(mol):\n    """{element: count} incl. total H (implicit + explicit) of an RDKit mol."""\n    counts = {}\n    for a in mol.GetAtoms():\n        s = a.GetSymbol()\n        counts[s] = counts.get(s, 0) + 1\n        h = a.GetTotalNumHs()\n        if h:\n            counts["H"] = counts.get("H", 0) + h\n    return counts\n\n\ndef ik14(smiles):\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return None\n    k = Chem.MolToInchiKey(m)\n    return k[:14] if k else None\n\n\ndef prep_structure(smiles, elements=ALLOWED_ELEMENTS):\n    """-> (status, canonical_smiles, selfies, formula). status \'ok\' or the reason it is excluded:\n    \'parse\', \'multi\' (more than one component), \'element\', \'encode\' (selfies constraints)."""\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return "parse", None, None, None\n    if len(Chem.GetMolFrags(m)) > 1:\n        return "multi", None, None, None\n    if any(a.GetSymbol() not in elements for a in m.GetAtoms()):\n        return "element", None, None, None\n    Chem.RemoveStereochemistry(m)\n    for a in m.GetAtoms():\n        a.SetIsotope(0)\n    can = Chem.MolToSmiles(m)\n    m = Chem.MolFromSmiles(can)\n    try:\n        s = sf.encoder(can, strict=True)\n    except Exception:\n        return "encode", can, None, None\n    if s is None:\n        return "encode", can, None, None\n    return "ok", can, s, mol_formula(m)\n\n\ndef reference_state(tokens):\n    """Slow truth for a token prefix: (element counts incl. explicit H atoms, rings + pi bonds)."""\n    Chem = _chem()\n    smi = sf.decoder("".join(tokens))\n    if not smi:\n        return {}, 0\n    m = Chem.MolFromSmiles(smi, sanitize=False)\n    counts = {}\n    for a in m.GetAtoms():\n        counts[a.GetSymbol()] = counts.get(a.GetSymbol(), 0) + 1\n    order = sum(int(b.GetBondTypeAsDouble()) for b in m.GetBonds())\n    n_comp = len(Chem.GetMolFrags(m, sanitizeFrags=False))\n    return counts, order - m.GetNumAtoms() + n_comp\n\n\ndef tracker_parity(table, tokens):\n    """Fast vs reference at every prefix. -> list of (prefix length, what, fast, reference)."""\n    vocab = table.vocab\n    t = Tracker(table)\n    bad = []\n    for k, tok in enumerate(tokens, 1):\n        t = t.push(vocab.tok2id[tok])\n        ref_counts, ref_u = reference_state(tokens[:k])\n        if t.element_counts() != ref_counts:\n            bad.append((k, "counts", t.element_counts(), ref_counts))\n        if t.pending is None and t.unsat() != ref_u:\n            bad.append((k, "unsat", t.unsat(), ref_u))\n        if t.broken:\n            bad.append((k, "broken", None, None))\n    return bad\n\n\ndef walk_truth(table, tokens, formula, max_len=None, dbe_prune=False):\n    """Mask soundness for one truth. -> list of (position, rule, token) violations.\n\n    rules: \'count\' (atom masked, element exhausted), \'grammar\' (masked for any other reason),\n    \'eos\' (</s> not legal after the last token), \'length\' (> max_len), \'dbe\' (unsat above DBE)."""\n    vocab = table.vocab\n    t = Tracker(table, formula)\n    limit = dbe(formula) if dbe_prune and dbe_prune_applies(formula) else None\n    bad = []\n    for k, tok in enumerate(tokens):\n        i = vocab.tok2id.get(tok, vocab.unk_id)\n        if not t.legal_mask()[i]:\n            r = t.remaining()\n            count_rule = (table.kind[i] == K_ATOM and t.pending is None and r is not None\n                          and r[table.elem_safe[i]] <= 0)\n            bad.append((k, "count" if count_rule else "grammar", tok))\n        t = t.push(i)\n        if limit is not None and t.pending is None and t.unsat() > limit + 1e-9:\n            bad.append((k, "dbe", tok))\n    if not t.eos_legal():\n        bad.append((len(tokens), "eos", None))\n    if max_len is not None and len(tokens) > max_len:\n        bad.append((len(tokens), "length", None))\n    return bad\n'

SRC = Path("/kaggle/working/novel_src")
SRC.mkdir(parents=True, exist_ok=True)
for fname, text in [("dreams_infer.py", DREAMS_INFER_SRC), ("novel_decoder.py", NOVEL_DECODER_SRC)]:
    got = hashlib.sha1(text.encode("utf-8")).hexdigest()
    assert got == MAN["src_sha1"][fname], f"{fname}: notebook source {got} != Colab's {MAN['src_sha1'][fname]}"
    (SRC / fname).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))
import dreams_infer as di
import novel_decoder as nd
RESULT["src_sha1_ok"] = True

## G0.1 — `dreams_embed.pt` on `cuda:1` vs Colab's vendored vectors

In [ ]:
n_gpu = torch.cuda.device_count()
DEV = "cuda:1" if n_gpu >= 2 else ("cuda:0" if n_gpu == 1 else "cpu")
RESULT["device"] = DEV
RESULT["n_gpu"] = n_gpu

PT = UP / "dreams_embed.pt"
t0 = time.time()
model, meta = di.load_dreams(PT, device=DEV, expect_sha1=MAN["dreams_embed_sha1"])
RESULT["sha1_ok"] = True
RESULT["t_load_s"] = round(time.time() - t0, 2)
print(f"loaded on {DEV} in {RESULT['t_load_s']}s; meta {meta}")

Z = np.load(UP / "parity_200.npz")
off, prec = Z["off"], Z["prec"]
PLS = [np.stack([Z["mz"][off[i]:off[i + 1]], Z["it"][off[i]:off[i + 1]]]).astype(np.float32)
       for i in range(len(prec))]
di.dreams_embed(model, PLS[:8], prec[:8])                  # warm-up (CUDA context, kernels)
if DEV.startswith("cuda"):
    torch.cuda.synchronize(DEV)
t0 = time.time()
EMB = di.dreams_embed(model, PLS, prec, batch_size=64)
if DEV.startswith("cuda"):
    torch.cuda.synchronize(DEV)
t_embed = time.time() - t0

def rowcos(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    return (x * y).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1))

cos_b = rowcos(EMB, Z["emb_b"])
cos_a = rowcos(EMB, Z["emb_a"])
cos_c = rowcos(EMB, Z["emb_c"])
RESULT.update(n_parity=len(PLS), cos_min=float(cos_b.min()), cos_median=float(np.median(cos_b)),
              cos_official_min=float(cos_a.min()), cos_cache_min=float(cos_c.min()),
              T_embed_ms_per_spectrum=1000 * t_embed / len(PLS), batch_size=64)
worst = np.argsort(cos_b)[:5]
print(f"Kaggle (b) vs Colab (b): min {cos_b.min():.6f}  median {np.median(cos_b):.6f}")
print(f"Kaggle (b) vs official (a): min {cos_a.min():.6f} | vs cache (c): min {cos_c.min():.6f}")
print(f"worst rows: {[(int(i), str(Z['adduct'][i]), float(cos_b[i])) for i in worst]}")
print(f"T_embed {RESULT['T_embed_ms_per_spectrum']:.2f} ms / spectrum on {DEV}")

## Tracker smoke (recorded, not gating)

In [ ]:
VOCAB = nd.SelfiesVocab.from_json(UP / "selfies_vocab.json")
TT = nd.TokenTable(VOCAB)
SMOKE = json.loads((UP / "tracker_smoke.json").read_text())
t0 = time.time()
viol = [nd.walk_truth(TT, list(sf.split_selfies(s["selfies"])), s["formula"], max_len=MAN["max_len"],
                      dbe_prune=MAN["dbe_prune"]) for s in SMOKE]
RESULT["tracker_smoke"] = dict(n=len(SMOKE), with_violations=int(sum(bool(v) for v in viol)),
                               seconds=round(time.time() - t0, 1))
print(RESULT["tracker_smoke"])

In [ ]:
passed = (RESULT["install_ok"] and RESULT["src_sha1_ok"] and RESULT["sha1_ok"] and DEV.startswith("cuda")
          and RESULT["cos_min"] >= MAN["cos_kaggle_min"])
RESULT["G0.1"] = "PASS" if passed else "FAIL"
RESULT["finished"] = time.strftime("%Y-%m-%d %H:%M")
Path("/kaggle/working/phase0_kaggle.json").write_text(json.dumps(RESULT, indent=1))
print("=" * 70)
print(json.dumps(RESULT, indent=1))
print("=" * 70)
print(f"G0.1 (Kaggle leg): {RESULT['G0.1']}   min cos {RESULT['cos_min']:.6f} (>= {MAN['cos_kaggle_min']})")